# Exp8a · Xu-inspired dual-adjacency GCN with a cortical-surface anatomical layer (amendment A2)

**Why this notebook exists.** On the recorded analysis atlas (lobes12) the original Exp 8 anatomical layer
(`lobe_hemisphere`: regions of the same lobe and hemisphere) is the zero matrix, because the 12 regions ARE the
lobe x hemisphere groups; every original Exp 8 unit failed before training. Amendment A2 (post-design, pre-result;
methodology log entry 60) redefines layer 2 as the fixed **cortical-surface contiguity** of the atlas regions
(fsaverage mesh, within a hemisphere): a data- and label-independent anatomical-neighbourhood prior, not measured
white-matter connectivity. Layer 1 is unchanged: the window's frequency-specific PLV or |ciPLV|. This is an
adaptation inspired by Xu et al. (2025), not a reproduction.

Config `configs/final/amendments/exp8a_xu_dual_gcn_surface.yaml` (`exp8a_xu_dual_gcn_surface`, 10 variants; `alt_bilateral_lobes` is an alternative
anatomical prior: each lobe linked to its contralateral homologue):

- `plv__psd_de_strength`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `ciplv__psd_de_strength`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `plv__psd`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `ciplv__psd`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `plv__de`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `ciplv__de`: layer 1 functional, layer 2 anatomical (surface_contiguity)
- `plv__psd_de_strength__abl_functional_both`: layer 1 functional, layer 2 functional
- `plv__psd_de_strength__abl_anatomical_both`: layer 1 anatomical, layer 2 anatomical (surface_contiguity)
- `plv__psd_de_strength__alt_bilateral_lobes`: layer 1 functional, layer 2 anatomical (lobe_bilateral)
- `plv__psd_de_strength__abl_shared_weights`: layer 1 functional, layer 2 anatomical (surface_contiguity), weights shared across bands

**Isolation.** This notebook checks out the amendment's own code pin
(`<outputs>/FINAL_v1/amendments/A2_exp8_surface/CODE_COMMIT.txt`) and writes only to `<outputs>/FINAL_v1/amendments/A2_exp8_surface/`.
The release pin, Exp 0-7, their results and the failed / partial units of the original Exp 8 are not touched or read
as checkpoints. Stores, QC reports and the atlas / graph decisions are read only. Same protocol (`gnn_fixed60_v1`),
folds and seeds; development CV only (the holdout is not run here).

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.
 Use a **GPU runtime** (T4).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move THIS AMENDMENT's code pin deliberately
                      # (<outputs>/FINAL_v1/amendments/A2_exp8_surface/CODE_COMMIT.txt; the release pin is never used or moved here).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--amendment', 'A2'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh as _sh

def sh(cmd, cwd=None):  # every script runs with the file backend: Colab's inline backend breaks figures in scripts
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)
A2_ROOT = FINAL + '/amendments/A2_exp8_surface'

## 1. Amendment check: pins, numerical environment, stores and QC (nothing is trained)

In [ ]:
ACCEPT_ENVIRONMENT_DIFFERENCE = False   # True only after reviewing a reported library difference (recorded)
sh("python scripts/final.py amendment-check --amendment A2 --atlas selected"
   + (" --accept-environment-difference" if ACCEPT_ENVIRONMENT_DIFFERENCE else ""))

## 2. The anatomical layer of the recorded atlas (fixed; from the atlas only)

In [ ]:
import json, numpy as np, pandas as pd
from adgraph.models.xu_dual import anatomical_adjacency
from adgraph.source.atlas import region_names
ATLAS_NAME = json.load(open(FINAL + '/decisions/atlas.json'))['atlas']
names = region_names(ATLAS_NAME)
A = anatomical_adjacency(names, 'surface_contiguity')
print(ATLAS_NAME, len(names), 'regions,', int(A.sum() // 2), 'surface edges, symmetric', bool((A == A.T).all()))
display(pd.DataFrame(A.astype(int), index=names, columns=names))

## 3. Contexts, status, run (resumable; this amendment experiment only)

In [ ]:
CONTEXTS = [('lcmv', 'B'), ('wmne', 'B'), ('lcmv', 'A'), ('wmne', 'A')]     # (inverse, window mode), in this order
ERRORS = {}
for inverse, mode in CONTEXTS:
    print(f'==== exp8a | {inverse} | mode {mode}')
    sh(f"python scripts/final.py status --amendment A2 --exp exp8a --inverse {inverse} --mode {mode} "
       f"--atlas selected --graph selected")
    try:
        sh(f"python scripts/final.py run --amendment A2 --exp exp8a --inverse {inverse} --mode {mode} "
           f"--atlas selected --graph selected")
    except RuntimeError as exc:
        ERRORS[(inverse, mode)] = exc
        print(f'{inverse} / mode {mode}: some units failed; details after the results')

## 4. Results and diagnostics of THIS amendment only (saved outputs; diagnostic figures)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
for inverse, mode in CONTEXTS:
    sh(f"python scripts/final.py report --amendment A2 --exp exp8a --inverse {inverse} --mode {mode} "
       f"--atlas selected --graph selected")
    rep = latest(f"{A2_ROOT}/results/{inverse}/mode_{mode}/{ATLAS_NAME}/*/exp8a_xu_dual_gcn_surface/_reports/report_*")
    if rep:
        print('====', inverse, 'mode', mode, rep)
        t = pd.read_csv(rep + '/results.csv')
        cols = ['variant', 'setting', 'auc', 'auc_ci_low', 'auc_ci_high', 'mean_fold_auc', 'min_fold_auc',
                'max_fold_auc', 'balanced_accuracy', 'sensitivity', 'specificity', 'tn', 'fp', 'fn', 'tp']
        display(t[[c for c in cols if c in t]].round(3))
        for png in sorted(glob.glob(rep + '/*.png')):
            display(Image(filename=png))
if ERRORS:
    raise RuntimeError(f'units did not complete in {sorted(ERRORS)}; completed units are saved. Run all again.')